In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "storageproject99")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/ciclo/ps_term_tbl.csv"

In [0]:
ciclo_schema = StructType(fields=[StructField("INSTITUTION", StringType(), False),
                                  StructField("ACAD_CAREER", StringType(), True),
                                  StructField("STRM", StringType(), True),
                                  StructField("DESCR", StringType(), True),
                                  StructField("DESCRSHORT", StringType(), True),
                                  StructField("TERM_BEGIN_DT", StringType(), True),
                                  StructField("TERM_END_DT", StringType(), True),
                                  StructField("WEEKS_OF_INSTRUCT", IntegerType(), True),
                                  StructField("TERM_CATEGORY", StringType(), True),
                                  StructField("ACAD_YEAR", IntegerType(), True)
                                  ])


In [0]:
df_ciclo_read = spark.read.option('header', True)\
                           .schema(ciclo_schema)\
                           .csv(ruta)

In [0]:
df_ciclo_read_dt_1 = df_ciclo_read.withColumn("TERM_BEGIN_DT",to_date(col("TERM_BEGIN_DT"), "d/MM/yyyy"))
df_ciclo_read_dt_2 = df_ciclo_read_dt_1.withColumn("TERM_END_DT",to_date(col("TERM_END_DT"), "d/MM/yyyy"))


In [0]:
df_ciclo_ingestion_date = df_ciclo_read_dt_2.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
 df_ciclo_final = df_ciclo_ingestion_date.select("INSTITUTION",
                                                 "ACAD_CAREER",
                                                 "STRM",
                                                 "DESCR",
                                                 "DESCRSHORT",
                                                 "TERM_BEGIN_DT",
                                                 "TERM_END_DT",
                                                 "WEEKS_OF_INSTRUCT",
                                                 "TERM_CATEGORY",
                                                 "ACAD_YEAR",
                                                 "INGESTION_DATE"
                                                 )

In [0]:
df_ciclo_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.ciclo")
